In [ ]:
# HODGE-ANCHORED CHANNEL-ENRICHED GLUEBALL KRYLOV SOLVER v2
# =========================================================
# Single self-contained Colab block.
#
# PURPOSE
# -------
# Return directly to the T1^{+-} pole calculation and enlarge the converged
# square-free graph backbone by the exact local SU(3) Feshbach channels already
# certified in the project.
#
# Retained P space:
#   connected square-free single fundamental Wilson loops generated from the
#   k=0, C=- plaquette source to graph depth K.
#
# Explicit local Q channels added to P:
#
#   1) one-shared-link like orientation
#        F x F = bar{3} + 6
#        weights = 1/3, 2/3
#        Delta E = 2, 3
#
#   2) one-shared-link opposite orientation
#        F x Fbar = 1 + 8
#        singlet weight = 1/9, adjoint weight = 8/9
#        Delta E = 4/3, 17/6
#      The singlet channel is already the retained square-free merged loop
#      whenever that loop is in P; only the orthogonal adjoint channel is then
#      added to Q. If the merged loop is outside P, its singlet channel is kept
#      explicitly in Q instead of being discarded.
#
#   3) exact SU(3) same-plaquette determinant decomposition in C=- sector
#
#        (chi_3 + chi_3bar) (chi_3 - chi_3bar)
#          = (chi_6 - chi_6bar) - (chi_3 - chi_3bar)
#
#      Since W = -M:
#        P W P = +P
#        <6,C-|W|3,C-> = -1
#        E_6 = 20/3.
#
# Thus the code does NOT patch a perturbative coefficient onto the old pole.
# It constructs an enlarged sparse Hamiltonian
#
#        H(y) = [ H_PP(y)    y B ]
#               [ y B^T      H_Q ]
#
# and follows the source pole of this enlarged Hilbert space nonperturbatively.
#
# CRITICAL COLD REGRESSION
# ------------------------
# Before reporting any mass/pole, the assembled P+Q model is Schur-reduced at
# E0=8/3 onto the one-plaquette sector.  All twelve shared-edge one-plaquette
# hoppings must reproduce
#
#        t_3 = 5/612
#
# with the exact signed incidence orientation.  This tests the actual channel
# weights, energies, C-odd signs, and graph couplings used by the pole solver.
#
# SCOPE
# -----
# This is the first channel-enriched production mass calculation.  It includes
# the certified LOCAL one-shared-link Feshbach sectors and the exact elementary
# SU(3) determinant/sextet complement.  It does not yet include every possible
# multi-link/multi-occupancy channel of arbitrary large loops, nor the string
# sector needed for M/sqrt(sigma).
#
# Recommended first run: MAX_K=2.  On L=3 this is only 2,889 P graphs but about
# 2e5 explicit local Q-channel states.  A CUDA GPU is used automatically for
# Lanczos if CuPy is available; CPU works too.

import math
import time
from collections import Counter, defaultdict, deque

import numpy as np
from scipy.linalg import eigh_tridiagonal
from scipy.sparse import coo_matrix, csr_matrix, diags

# =============================================================================
# CONFIG
# =============================================================================
N = 3
L = 3
MAX_K = 2
Y_VALUES = (0.02, 0.05, 0.10, 0.20, 0.30)
POLARIZATION = (0, 1)  # xy component; the other T1 components are symmetry-related

PREFER_GPU = True
MAX_LANCZOS = 180
RITZ_TOL = 2e-10
ENERGY_STABILITY_TOL = 2e-11
RESIDUE_STABILITY_TOL = 2e-10
STABLE_CHECKS_REQUIRED = 2

CF = (N * N - 1) / (2 * N)   # 4/3
E0_PLAQ = 2 * CF             # 8/3
T3_TARGET = 5.0 / 612.0

# Exact SU(3) local Casimirs and decomposition weights.
C_BAR3 = 4.0 / 3.0
C_6 = 10.0 / 3.0
C_8 = 3.0

W_A = 1.0 / 3.0
W_S = 2.0 / 3.0
W_1 = 1.0 / 9.0
W_8 = 8.0 / 9.0

D_A = CF + 0.5 * C_BAR3     # 2
D_S = CF + 0.5 * C_6        # 3
D_1 = CF                     # 4/3
D_8 = CF + 0.5 * C_8        # 17/6
E_SEXTET_PLAQ = 2.0 * C_6   # 20/3

# =============================================================================
# OPTIONAL CUDA BACKEND FOR LANCZOS
# =============================================================================
USE_GPU = False
cp = None
cpsparse = None

if PREFER_GPU:
    try:
        import cupy as _cp
        import cupyx.scipy.sparse as _cpsparse
        if int(_cp.cuda.runtime.getDeviceCount()) > 0:
            cp = _cp
            cpsparse = _cpsparse
            USE_GPU = True
    except Exception:
        USE_GPU = False

xp = cp if USE_GPU else np
DEVICE = "CUDA/CuPy" if USE_GPU else "CPU/NumPy"

# =============================================================================
# GATES
# =============================================================================
gates = []

def gate(name, ok, detail=""):
    ok = bool(ok)
    gates.append((name, ok, str(detail)))
    print(("[PASS] " if ok else "[FAIL] ") + name + (f" :: {detail}" if detail else ""))

# =============================================================================
# CUBIC CELL COMPLEX
# =============================================================================
def shift(v, d, step=1):
    w = list(v)
    w[d] = (w[d] + step) % L
    return tuple(w)


def build_cubic_complex():
    verts = [(x, y, z) for x in range(L) for y in range(L) for z in range(L)]

    links = []
    lid = {}
    for v in verts:
        for d in range(3):
            lid[(v, d)] = len(links)
            links.append((v, d))

    faces = []
    for v in verts:
        for a, b in ((0, 1), (0, 2), (1, 2)):
            faces.append((v, a, b))

    B2 = np.zeros((len(links), len(faces)), dtype=np.int8)
    for f, (v, a, b) in enumerate(faces):
        va = shift(v, a)
        vb = shift(v, b)
        B2[lid[(v, a)], f] += 1
        B2[lid[(va, b)], f] += 1
        B2[lid[(vb, a)], f] -= 1
        B2[lid[(v, b)], f] -= 1

    return verts, links, faces, B2


verts, links, faces, B2 = build_cubic_complex()
E, P = B2.shape

link_faces = [[] for _ in range(E)]
for f in range(P):
    for l in np.flatnonzero(B2[:, f]):
        link_faces[int(l)].append(f)

source_faces = [
    f for f, (_, a, b) in enumerate(faces)
    if (a, b) == POLARIZATION
]

# =============================================================================
# C-ODD GRAPH ALGEBRA
# =============================================================================
def canonical_codd(q):
    q = np.asarray(q, dtype=np.int8)
    a = q.tobytes()
    b = (-q).tobytes()
    if a <= b:
        return a, +1, q.copy()
    return b, -1, (-q).copy()


def canonical_pair_codd(q, b):
    """Canonical unordered two-trace product under global charge conjugation."""
    q = np.asarray(q, dtype=np.int8)
    b = np.asarray(b, dtype=np.int8)
    pair = tuple(sorted((q.tobytes(), b.tobytes())))
    pair_c = tuple(sorted(((-q).tobytes(), (-b).tobytes())))
    if pair <= pair_c:
        return pair, +1
    return pair_c, -1


def is_simple_single_loop(q):
    support = np.flatnonzero(q)
    if len(support) < 4:
        return False

    outgoing = {}
    indegree = Counter()
    used = set()

    for li in support:
        l = int(li)
        v, d = links[l]
        w = shift(v, d)
        if q[l] > 0:
            src, dst = v, w
        else:
            src, dst = w, v
        if src in outgoing:
            return False
        outgoing[src] = dst
        indegree[dst] += 1
        used.add(src)
        used.add(dst)

    if any(v not in outgoing or indegree[v] != 1 for v in used):
        return False

    start = next(iter(used))
    cur = start
    seen = set()
    while cur not in seen:
        seen.add(cur)
        cur = outgoing[cur]

    return cur == start and len(seen) == len(used)


def candidate_faces(q):
    out = set()
    for l in np.flatnonzero(q):
        out.update(link_faces[int(l)])
    return out


def plaquette_match(q):
    for f in range(P):
        if np.array_equal(q, B2[:, f]):
            return f, +1
        if np.array_equal(q, -B2[:, f]):
            return f, -1
    return None

# =============================================================================
# SQUARE-FREE P BASIS
# =============================================================================
def build_graph_basis(max_depth):
    states = {}
    depth = {}
    queue = deque()
    source_terms = []

    for f in source_faces:
        key, sign, rep = canonical_codd(B2[:, f])
        if key not in states:
            states[key] = rep
            depth[key] = 0
            queue.append(key)
        source_terms.append((key, sign))

    while queue:
        key = queue.popleft()
        d = depth[key]
        if d >= max_depth:
            continue

        q = states[key]
        for f in candidate_faces(q):
            bf = B2[:, f]
            for s in (-1, +1):
                z = q + s * bf
                if np.max(np.abs(z)) > 1:
                    continue
                if not is_simple_single_loop(z):
                    continue
                key2, _, rep2 = canonical_codd(z)
                if key2 not in states:
                    states[key2] = rep2
                    depth[key2] = d + 1
                    queue.append(key2)

    keys = list(states)
    index = {k: i for i, k in enumerate(keys)}
    reps = [states[k] for k in keys]

    source = np.zeros(len(keys), dtype=np.float64)
    for key, sign in source_terms:
        source[index[key]] += sign / math.sqrt(len(source_terms))

    if abs(np.dot(source, source) - 1.0) > 1e-12:
        raise RuntimeError("source normalization failed")

    return keys, reps, depth, index, source

# =============================================================================
# CHANNEL-ENRICHED P + Q HAMILTONIAN
# =============================================================================
def build_enriched_model(max_depth):
    t0 = time.time()
    keys, states, depth, index, source = build_graph_basis(max_depth)
    nP = len(states)

    E_P = np.asarray([
        0.5 * int(np.count_nonzero(q)) * CF
        for q in states
    ], dtype=np.float64)

    pp_rows, pp_cols, pp_vals = [], [], []
    p_diag = np.zeros(nP, dtype=np.float64)

    q_index = {}
    q_energy = []
    q_type = []
    pq_rows, pq_cols, pq_vals = [], [], []

    def get_q(key, energy, typ):
        if key in q_index:
            j = q_index[key]
            if abs(q_energy[j] - energy) > 1e-10:
                raise RuntimeError(
                    f"inconsistent Q energy for {typ}: {q_energy[j]} vs {energy}"
                )
            return j
        j = len(q_energy)
        q_index[key] = j
        q_energy.append(float(energy))
        q_type.append(str(typ))
        return j

    for i, q in enumerate(states):
        perimeter = int(np.count_nonzero(q))
        E_parent = E_P[i]

        # Exact same-plaquette SU(3) determinant decomposition in C=- sector:
        # W |3,C-> = +|3,C-> - |6,C->.
        pm = plaquette_match(q) if perimeter == 4 else None
        if pm is not None:
            p_diag[i] += 1.0
            qkey = ("sextet_plaquette", canonical_codd(q)[0])
            j = get_q(qkey, E_SEXTET_PLAQ, "sextet_plaquette")
            pq_rows.append(i)
            pq_cols.append(j)
            pq_vals.append(-1.0)

        for f in candidate_faces(q):
            bf = B2[:, f]
            shared = np.flatnonzero((q != 0) & (bf != 0))

            for s in (-1, +1):
                b = (s * bf).astype(np.int8)
                z = q + b

                # Retained fundamental-loop deformation.  This is the singlet
                # branch of a one-link F x Fbar overlap when one shared edge is
                # cancelled, but is kept directly in P because the resulting
                # square-free loop is an explicit graph state.
                retained_loop = (
                    np.max(np.abs(z)) <= 1
                    and is_simple_single_loop(z)
                )
                if retained_loop:
                    key2, codd_sign, _ = canonical_codd(z)
                    j = index.get(key2)
                    if j is not None and j != i:
                        pp_rows.append(i)
                        pp_cols.append(j)
                        pp_vals.append(-codd_sign / N)  # W=-M

                # The local representation decomposition below is certified only
                # for exactly one shared physical link.  Multi-link overlaps are
                # deliberately not assigned an invented channel amplitude.
                if len(shared) != 1:
                    continue

                l = int(shared[0])
                relation = int(q[l]) * int(b[l])
                pair_key, q_codd_sign = canonical_pair_codd(q, b)

                if relation > 0:
                    # F x F = bar3 + 6.
                    for typ, weight, delta in (
                        ("like_bar3", W_A, D_A),
                        ("like_6", W_S, D_S),
                    ):
                        j = get_q((typ, pair_key), E_parent + delta, typ)
                        pq_rows.append(i)
                        pq_cols.append(j)
                        pq_vals.append(-q_codd_sign * math.sqrt(weight))

                else:
                    # F x Fbar = 1 + 8.
                    # If the singlet merged loop is explicitly in P, the exact
                    # -1/N P-P matrix element above already represents it.  Keep
                    # only the orthogonal adjoint channel in Q.  If truncation
                    # excludes the merged loop, retain its singlet Q channel too.
                    keyz = canonical_codd(z)[0] if retained_loop else None
                    singlet_is_in_P = retained_loop and keyz in index

                    if not singlet_is_in_P:
                        typ = "mixed_1"
                        j = get_q((typ, pair_key), E_parent + D_1, typ)
                        pq_rows.append(i)
                        pq_cols.append(j)
                        pq_vals.append(-q_codd_sign * math.sqrt(W_1))

                    typ = "mixed_8"
                    j = get_q((typ, pair_key), E_parent + D_8, typ)
                    pq_rows.append(i)
                    pq_cols.append(j)
                    pq_vals.append(-q_codd_sign * math.sqrt(W_8))

    W_PP = coo_matrix(
        (pp_vals, (pp_rows, pp_cols)),
        shape=(nP, nP),
        dtype=np.float64,
    ).tocsr()
    W_PP.sum_duplicates()
    W_PP = W_PP + diags(p_diag, dtype=np.float64)

    asym = W_PP - W_PP.T
    asymmetry = 0.0 if asym.nnz == 0 else float(np.max(np.abs(asym.data)))
    if asymmetry > 1e-12:
        raise RuntimeError(f"P-space Hamiltonian asymmetry = {asymmetry}")

    nQ = len(q_energy)
    B_PQ = coo_matrix(
        (pq_vals, (pq_rows, pq_cols)),
        shape=(nP, nQ),
        dtype=np.float64,
    ).tocsr()
    B_PQ.sum_duplicates()

    return {
        "keys": keys,
        "states": states,
        "depth": depth,
        "index": index,
        "source_P": source,
        "E_P": E_P,
        "W_PP": W_PP,
        "B_PQ": B_PQ,
        "E_Q": np.asarray(q_energy, dtype=np.float64),
        "Q_types": q_type,
        "asymmetry": asymmetry,
        "build_seconds": time.time() - t0,
    }

# =============================================================================
# COLD t3 REGRESSION FROM THE ACTUAL ENRICHED MODEL
# =============================================================================
def validate_second_order_t3(model):
    states = model["states"]
    index = model["index"]
    E_P = model["E_P"]
    E_Q = model["E_Q"]
    W_PP = model["W_PP"]
    B_PQ = model["B_PQ"]

    plaquette_mask = np.asarray([
        plaquette_match(q) is not None for q in states
    ], dtype=bool)

    seed_face = 0
    pkey, psgn, _ = canonical_codd(B2[:, seed_face])
    if pkey not in index:
        raise RuntimeError("seed plaquette absent from P basis")
    ip = index[pkey]

    p_row = {
        int(k): float(v)
        for k, v in zip(W_PP.getrow(ip).indices, W_PP.getrow(ip).data)
    }
    q_row = {
        int(k): float(v)
        for k, v in zip(B_PQ.getrow(ip).indices, B_PQ.getrow(ip).data)
    }

    errors = []
    rows = []

    for qf in range(P):
        if qf == seed_face:
            continue
        shared = np.flatnonzero(
            (B2[:, seed_face] != 0) & (B2[:, qf] != 0)
        )
        if len(shared) != 1:
            continue

        qkey, qsgn, _ = canonical_codd(B2[:, qf])
        if qkey not in index:
            continue
        iq = index[qkey]

        p2 = {
            int(k): float(v)
            for k, v in zip(W_PP.getrow(iq).indices, W_PP.getrow(iq).data)
        }
        q2 = {
            int(k): float(v)
            for k, v in zip(B_PQ.getrow(iq).indices, B_PQ.getrow(iq).data)
        }

        coeff = 0.0

        for k in set(p_row).intersection(p2):
            if plaquette_mask[k]:
                continue
            coeff += p_row[k] * p2[k] / (E0_PLAQ - E_P[k])

        for k in set(q_row).intersection(q2):
            coeff += q_row[k] * q2[k] / (E0_PLAQ - E_Q[k])

        l = int(shared[0])
        incidence = int(B2[l, seed_face] * B2[l, qf])

        # Matrix entries are written in the canonical C-odd representatives,
        # hence the extra p/q canonical signs relative to the raw face basis.
        target = psgn * qsgn * incidence * T3_TARGET
        err = coeff - target
        errors.append(abs(err))
        rows.append((qf, incidence, qsgn, coeff, target, err))

    if len(rows) != 12:
        raise RuntimeError(f"expected 12 shared-edge plaquette neighbors, got {len(rows)}")

    gate(
        "enriched P+Q Hamiltonian cold-reproduces all 12 t3 hoppings",
        max(errors) < 5e-13,
        f"max |cold-target|={max(errors):.3e}",
    )

    return rows

# =============================================================================
# RUNTIME BACKEND
# =============================================================================
def prepare_runtime(model):
    nP = len(model["E_P"])
    nQ = len(model["E_Q"])

    source = np.zeros(nP + nQ, dtype=np.float64)
    source[:nP] = model["source_P"]

    if USE_GPU:
        E_P_b = cp.asarray(model["E_P"])
        E_Q_b = cp.asarray(model["E_Q"])
        W_PP_b = cpsparse.csr_matrix(model["W_PP"])
        B_b = cpsparse.csr_matrix(model["B_PQ"])
        BT_b = B_b.T.tocsr()
        source_b = cp.asarray(source)
    else:
        E_P_b = model["E_P"]
        E_Q_b = model["E_Q"]
        W_PP_b = model["W_PP"]
        B_b = model["B_PQ"]
        BT_b = B_b.T.tocsr()
        source_b = source

    return {
        "nP": nP,
        "nQ": nQ,
        "E_P": E_P_b,
        "E_Q": E_Q_b,
        "W_PP": W_PP_b,
        "B": B_b,
        "BT": BT_b,
        "source": source_b,
    }

# =============================================================================
# RESIDUAL-CONTROLLED SOURCE LANCZOS
# =============================================================================
def _to_float(x):
    if USE_GPU:
        return float(cp.asnumpy(x).reshape(()))
    return float(x)


def source_lanczos(runtime, y, enriched=True):
    nP = runtime["nP"]
    nQ = runtime["nQ"] if enriched else 0
    n = nP + nQ

    E_P = runtime["E_P"]
    E_Q = runtime["E_Q"]
    W_PP = runtime["W_PP"]
    B = runtime["B"]
    BT = runtime["BT"]

    if enriched:
        source = runtime["source"]
    else:
        source = runtime["source"][:nP]

    q = source / xp.linalg.norm(source)
    q_prev = xp.zeros_like(q)
    beta_prev = 0.0

    # Preallocate the reorthogonalization basis.  K=2 enriched is ~2e5 states;
    # 180 float64 vectors are ~290 MiB, comfortable on an A100 and typical Colab RAM.
    Q = xp.empty((MAX_LANCZOS, n), dtype=xp.float64)

    alpha = []
    beta = []
    last_pole = None
    stable_count = 0
    last_eval = None
    last_weight = None
    last_residual = float("inf")

    def matvec(x):
        p = x[:nP]
        out_p = E_P * p + y * (W_PP @ p)
        if not enriched:
            return out_p

        qpart = x[nP:]
        out_p = out_p + y * (B @ qpart)
        out_q = E_Q * qpart + y * (BT @ p)
        return xp.concatenate((out_p, out_q))

    for m in range(MAX_LANCZOS):
        Q[m] = q
        z = matvec(q)

        if m:
            z -= beta_prev * q_prev

        a = _to_float(xp.dot(q, z))
        z -= a * q

        # Two-pass full reorthogonalization, but performed as GEMV/GEMM against
        # the already-built Krylov slab rather than Python loops over vectors.
        Qm = Q[:m + 1]
        for _ in range(2):
            coeff = Qm @ z
            z -= coeff @ Qm

        b = _to_float(xp.linalg.norm(z))
        alpha.append(a)

        # T_{m+1} uses all PREVIOUS beta values.  The current b is the residual
        # coupling from the current tridiagonal space to the next Lanczos vector.
        if m >= 2:
            evals, evecs = eigh_tridiagonal(
                np.asarray(alpha, dtype=float),
                np.asarray(beta, dtype=float),
            )
            weights = evecs[0, :] ** 2
            pole = int(np.argmax(weights))
            E_pole = float(evals[pole])
            Z_pole = float(weights[pole])
            ritz_residual = float(b * abs(evecs[-1, pole]))

            if last_pole is not None:
                dE = abs(E_pole - last_pole[0])
                dZ = abs(Z_pole - last_pole[1])
                stable = (
                    ritz_residual < RITZ_TOL
                    and dE < ENERGY_STABILITY_TOL
                    and dZ < RESIDUE_STABILITY_TOL
                )
                stable_count = stable_count + 1 if stable else 0
            else:
                stable_count = 0

            last_pole = (E_pole, Z_pole)
            last_eval = evals
            last_weight = weights
            last_residual = ritz_residual

            if stable_count >= STABLE_CHECKS_REQUIRED:
                return {
                    "pole_energy": E_pole,
                    "pole_residue": Z_pole,
                    "ritz_residual": ritz_residual,
                    "krylov_dimension": m + 1,
                    "converged": True,
                }

        if b < 1e-14:
            # Exact/near-exact Krylov closure.  The current Ritz pair is converged
            # even if the stability counter did not have another iteration to fire.
            if last_pole is not None:
                return {
                    "pole_energy": float(last_pole[0]),
                    "pole_residue": float(last_pole[1]),
                    "ritz_residual": float(last_residual),
                    "krylov_dimension": m + 1,
                    "converged": True,
                }
            break

        if m < MAX_LANCZOS - 1:
            beta.append(b)
        q_prev = q
        q = z / b
        beta_prev = b

    if last_pole is None:
        # Tiny one-dimensional cases.
        return {
            "pole_energy": float(alpha[0]),
            "pole_residue": 1.0,
            "ritz_residual": 0.0,
            "krylov_dimension": 1,
            "converged": True,
        }

    return {
        "pole_energy": float(last_pole[0]),
        "pole_residue": float(last_pole[1]),
        "ritz_residual": float(last_residual),
        "krylov_dimension": len(alpha),
        "converged": False,
    }

# =============================================================================
# RUN
# =============================================================================
print("=" * 112)
print("HODGE-ANCHORED CHANNEL-ENRICHED T1^{+-} GLUEBALL POLE")
print("=" * 112)
print(f"backend            : {DEVICE}")
print(f"SU(N)              : SU({N})")
print(f"periodic lattice   : {L}^3")
print(f"graph depth max    : K={MAX_K}")
print(f"source polarization: {POLARIZATION}")
print(f"C_F                : {CF:.12g}")
print()

# Local exact algebra gates.
gate("SU(3) F x F weights sum to one", abs((W_A + W_S) - 1.0) < 1e-15)
gate("SU(3) F x Fbar weights sum to one", abs((W_1 + W_8) - 1.0) < 1e-15)
gate("adjacent like electric gaps are {2,3}", abs(D_A - 2.0) < 1e-15 and abs(D_S - 3.0) < 1e-15)
gate("adjacent mixed electric gaps are {4/3,17/6}", abs(D_1 - 4/3) < 1e-15 and abs(D_8 - 17/6) < 1e-15)
gate("elementary sextet C-odd channel energy is 20/3", abs(E_SEXTET_PLAQ - 20/3) < 1e-15)

results = {}

for K in range(MAX_K + 1):
    print("\n" + "-" * 112)
    print(f"BUILDING CHANNEL-ENRICHED MODEL: K={K}")
    model = build_enriched_model(K)

    nP = len(model["E_P"])
    nQ = len(model["E_Q"])
    qhist = Counter(model["Q_types"])
    phist = Counter(int(np.count_nonzero(q)) for q in model["states"])

    print(f"P graph states      : {nP:,}")
    print(f"Q Feshbach states   : {nQ:,}")
    print(f"total Hilbert dim   : {nP+nQ:,}")
    print(f"build time          : {model['build_seconds']:.3f} s")
    print(f"P-space asymmetry   : {model['asymmetry']:.3e}")
    print(f"P perimeter spectrum: {dict(sorted(phist.items()))}")
    print(f"Q channel counts    : {dict(qhist)}")

    if K == MAX_K and K >= 2:
        rows = validate_second_order_t3(model)
        maxerr = max(abs(r[-1]) for r in rows)
        print(f"cold t3 regression : 12/12 neighbors; max error={maxerr:.3e}")

    runtime = prepare_runtime(model)
    results[K] = {}

    for y in Y_VALUES:
        sf = source_lanczos(runtime, y, enriched=False)
        en = source_lanczos(runtime, y, enriched=True)
        results[K][y] = {"squarefree": sf, "enriched": en}

        print(
            f"  y={y:5.2f}  "
            f"E_SF={sf['pole_energy']:.10f}  "
            f"E_ENR={en['pole_energy']:.10f}  "
            f"dE={en['pole_energy']-sf['pole_energy']:+.6e}  "
            f"Z_ENR={en['pole_residue']:.8f}  "
            f"Ritz={en['ritz_residual']:.2e}  "
            f"m={en['krylov_dimension']:3d}  "
            f"{'OK' if en['converged'] else 'CAP'}"
        )

# Central derivative at the deepest K checks the exact PWP=+P sign without a
# finite-y forward-secant contamination.
print("\n" + "=" * 112)
print("DEEPEST-K POLE SUMMARY")
print("=" * 112)
K = MAX_K
for y in Y_VALUES:
    en = results[K][y]["enriched"]
    print(
        f"y={y:5.2f}  M_trunc={en['pole_energy']:.10f}  "
        f"Z={en['pole_residue']:.8f}  Ritz={en['ritz_residual']:.3e}"
    )

# A very small +/-epsilon check at K=0 is enough to test the exact local
# first-order determinant sign while keeping runtime negligible.
model0 = build_enriched_model(0)
runtime0 = prepare_runtime(model0)
eps = 1e-4
Ep = source_lanczos(runtime0, +eps, enriched=True)["pole_energy"]
Em = source_lanczos(runtime0, -eps, enriched=True)["pole_energy"]
central_slope = (Ep - Em) / (2 * eps)
gate("central y-derivative of enriched pole is +1 at strong coupling",
     abs(central_slope - 1.0) < 2e-6,
     f"dE/dy={central_slope:.12g}")

print("\n" + "=" * 112)
print("FINAL GATE SUMMARY")
print("=" * 112)
passed = sum(ok for _, ok, _ in gates)
for i, (name, ok, detail) in enumerate(gates, 1):
    print(f"{i:02d}. {'PASS' if ok else 'FAIL'} — {name}" + (f" :: {detail}" if detail else ""))
print("-" * 112)
print(f"PASSED {passed}/{len(gates)} GATES")

if passed != len(gates):
    raise AssertionError("channel-enriched mass solver failed an exact local regression gate")

if any(not results[MAX_K][y]["enriched"]["converged"] for y in Y_VALUES):
    print("WARNING: at least one deepest-K pole reached MAX_LANCZOS before the Ritz criterion.")
    print("Increase MAX_LANCZOS; do not promote uncaught final digits.")
else:
    print("All deepest-K source poles satisfy the Ritz residual/stability criterion.")

print("\nRESULT")
print("------")
print("The table above is the channel-enriched T1^{+-} source pole of the explicit P+Q Hamiltonian.")
print("It is the mass calculation in the current controlled Hilbert-space truncation, not a fitted series.")
print("The next physical observable required for a continuum comparison is the same-Hamiltonian winding-string sector sigma(y).")
